# Segmental duplications per chromosome

Per-chromosome summary of segmental duplications (BISER `segdup_output_duplicateLinkRemoved.bedpe`):
intra- vs inter-chromosomal link counts, alignment scores, merged coverage, and the
percentage of each chromosome covered by segdups.


In [ ]:
# Project root — edit for your environment.
PROJ_ROOT = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj"


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from natsort import natsorted

In [ ]:
fai = pd.read_csv(
    f"{PROJ_ROOT}/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)
fai=fai[~fai["chr"].str.contains("seq")]
fai=fai[~fai["chr"].str.contains("NC")]


# Read BEDPE file
bedpe = pd.read_csv(
    f"{PROJ_ROOT}/code/command-line-script/genome-annotation/biser/hifiasm-041425/segdup_output_duplicateLinkRemoved.bedpe",
    sep="\t",
    header=None,
    names=["chr1", "start1", "end1", "chr2", "start2", "end2","reference","score","strand1","strand2","max_len","aln_len","cigar","optional"]
)

# bedpe = bedpe.sample(frac=0.1, random_state=28)  # random 1000 rows
df = bedpe.copy()
df=df[~df["chr1"].str.contains("seq")]
df=df[~df["chr2"].str.contains("seq")]
# df=df[~df["chr2"].str.contains("seq")]
# df=df[(df["aln_len"]>1000)]
# df=df[(df["aln_len"]>1000)&(df["score"]<=30)]

In [ ]:
# Create a new column to indicate intra vs inter chromosomal
df['link_type'] = df.apply(lambda row: 'Intrachromosomal' if row['chr1'] == row['chr2'] else 'Interchromosomal', axis=1)

## Flip just the interchromosomal links to properly count the interchromosomal links 
# Create copies of interchromosomal links with chr1/chr2 flipped
inter_links = df[df['link_type'] == 'Interchromosomal'].copy()

# Flip the coordinates for interchromosomal links
flipped_inter = inter_links.copy()
flipped_inter[['chr1', 'start1', 'end1', 'chr2', 'start2', 'end2']] = \
    inter_links[['chr2', 'start2', 'end2', 'chr1', 'start1', 'end1']].values

# Combine original with flipped version
flipped_df_for_counting = pd.concat([df, flipped_inter], ignore_index=True)

print(f"Original dataframe: {len(df)} rows")
print(f"Symmetric dataframe: {len(flipped_df_for_counting)} rows")
print(f"Added {len(flipped_inter)} flipped interchromosomal links")

In [ ]:
def merge_alignments(df, chrom_col='chr1', start_col='start1', end_col='end1', type_col='link_type'):
    """
    Merge overlapping or adjacent alignments of the same type on the same chromosome
    Returns a DataFrame with merged intervals and their total lengths
    """
    merged_data = []
    
    # Group by chromosome and alignment type
    for (chrom, aln_type), group in df.groupby([chrom_col, type_col]):
        # Sort by start position
        sorted_group = group.sort_values(by=[start_col, end_col])
        
        if len(sorted_group) == 0:
            continue
            
        merged_intervals = []
        # Initialize with first interval
        current_start = sorted_group.iloc[0][start_col]
        current_end = sorted_group.iloc[0][end_col]
        
        for i in range(1, len(sorted_group)):
            row = sorted_group.iloc[i]
            start, end = row[start_col], row[end_col]
            
            # Check for overlap or adjacency
            if start <= current_end:
                # Overlapping or adjacent - extend current interval if needed
                current_end = max(current_end, end)
            else:
                # No overlap - save current interval and start new one
                merged_intervals.append({
                    'chrom': chrom,
                    'aln_type': aln_type,
                    'start': current_start,
                    'end': current_end,
                    'length': current_end - current_start
                })
                current_start = start
                current_end = end
        
        # Don't forget the last interval
        merged_intervals.append({
            'chrom': chrom,
            'aln_type': aln_type,
            'start': current_start,
            'end': current_end,
            'length': current_end - current_start
        })
        
        merged_data.extend(merged_intervals)
    
    return pd.DataFrame(merged_data)
def calculate_merged_lengths_by_chromosome_simple(df):
    """
    Calculate total merged alignment length for each chromosome by segdup type
    Using only chr1 since we already flipped interchromosomal links
    """
    # Merge regions from chr1 for all segdup types
    merged_all = merge_alignments(df, chrom_col='chr1', start_col='start1', end_col='end1', type_col='link_type')
    
    # Sum by chromosome and link_type
    result = merged_all.groupby(['chrom', 'aln_type'])['length'].sum().reset_index()
    
    # Convert to Mbp for readability
    result['length_mbp'] = result['length'] / 1e6
    
    return result



In [ ]:
# Calculate the merged lengths using the simplified approach
merged_lengths = calculate_merged_lengths_by_chromosome_simple(flipped_df_for_counting)



In [ ]:
# Pivot merged lengths to a per-chromosome (intra vs inter) matrix for the summaries.
pivot_lengths = merged_lengths.pivot(index='chrom', columns='aln_type', values='length_mbp').fillna(0)


In [ ]:

# Create the side-by-side bar plot using seaborn
plt.figure(figsize=(10, 5))

# Create the bar plot
ax = sns.barplot(x='chr1', y='count', hue='link_type', 
                 data=flipped_df_for_counting.groupby(['chr1', 'link_type']).size().reset_index(name='count'),
                 palette={'Intrachromosomal': 'blue', 'Interchromosomal': 'red'},
                 order=natsorted(flipped_df_for_counting['chr1'].unique()))  # ADD THIS ORDER PARAMETER
# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# Add horizontal guide lines
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)  # Ensure grid lines are behind the bars
# Customize the plot
# plt.title('Number of Segmental Duplications by Chromosome and Link Type', fontsize=14, fontweight='bold')
plt.xlabel('Chromosome', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=45)
plt.legend(title='Segdup Link Type', title_fontsize=10)

# Add count labels on the bars
for i, container in enumerate(ax.containers):
    for j, bar in enumerate(container):
        height = bar.get_height()
        if height > 0:  # Only label bars with positive height
            ax.text(bar.get_x() + bar.get_width() / 2, height + (ax.get_ylim()[1] * 0.01),
                   f'{int(height)}', ha='center', va='bottom', fontsize=8, fontweight='bold',
                   color='black' if i == 0 else 'black', rotation=15)  # White text for better contrast

plt.tight_layout()
plt.show()

# Print summary statistics
print("Summary Statistics:")
print("="*60)
print(f"{'Chromosome':<10} {'Link Type':<15} {'Count':<8} {'Mean Score':<12} {'Std Dev':<10}")
print("-"*60)

for chrom in natsorted(flipped_df_for_counting['chr1'].unique()):
    chrom_data = flipped_df_for_counting[flipped_df_for_counting['chr1'] == chrom]
    for link_type in ['Intrachromosomal', 'Interchromosomal']:
        type_data = chrom_data[chrom_data['link_type'] == link_type]
        if len(type_data) > 0:
            print(f"{chrom:<10} {link_type:<15} {len(type_data):<8} {type_data['score'].mean():<12.2f} {type_data['score'].std():<10.2f}")

# Overall statistics
print(f"\nTotal intrachromosomal links: {len(flipped_df_for_counting[flipped_df_for_counting['link_type'] == 'Intrachromosomal'])}")
print(f"Total interchromosomal links: {len(flipped_df_for_counting[flipped_df_for_counting['link_type'] == 'Interchromosomal'])}")
print(f"Mean score - Intrachromosomal: {flipped_df_for_counting[flipped_df_for_counting['link_type'] == 'Intrachromosomal']['score'].mean():.2f}")
print(f"Mean score - Interchromosomal: {flipped_df_for_counting[flipped_df_for_counting['link_type'] == 'Interchromosomal']['score'].mean():.2f}")

In [ ]:


# Create a new column to indicate intra vs inter chromosomal
# df['link_type'] = df.apply(lambda row: 'Intrachromosomal' if row['chr1'] == row['chr2'] else 'Interchromosomal', axis=1)

# Create the violin plot
plt.figure(figsize=(10, 5))

# Create violin plot with swarm plot overlay
ax = sns.violinplot(x='chr1', y='score', hue='link_type', data=flipped_df_for_counting, 
                    palette={'Intrachromosomal': 'blue', 'Interchromosomal': 'red'},
                    split=True, inner='quartile',edgecolor="black")
              
# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# Add horizontal guide lines
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)  # Ensure grid lines are behind the bars
# Customize the plot
# plt.title('Distribution of Segmental Duplication Scores by Chromosome and Link Type', fontsize=14, fontweight='bold')
plt.xlabel('Chromosome', fontsize=12)
plt.ylabel('Total Alignement Error', fontsize=12)
plt.xticks(rotation=45)
plt.legend(title='Link Type', title_fontsize=10)
# ax.legend_.remove()

# Add some statistics to the plot
chromosomes = natsorted(df['chr1'].unique())
for i, chrom in enumerate(chromosomes):
    chrom_data = df[df['chr1'] == chrom]
    intra_data = chrom_data[chrom_data['link_type'] == 'Intrachromosomal']['score']
    inter_data = chrom_data[chrom_data['link_type'] == 'Interchromosomal']['score']
    
    # if len(intra_data) > 0:
    #     plt.text(i-0.2, intra_data.max() + 2, f'n={len(intra_data)}', 
    #             ha='center', va='bottom', fontsize=8, color='blue')
    # if len(inter_data) > 0:
    #     plt.text(i+0.2, inter_data.max() + 2, f'n={len(inter_data)}', 
    #             ha='center', va='bottom', fontsize=8, color='red')

plt.tight_layout()
plt.show()

# Print summary statistics
print("Summary Statistics:")
print("="*60)
print(f"{'Chromosome':<10} {'Link Type':<15} {'Count':<8} {'Mean Score':<12} {'Std Dev':<10}")
print("-"*60)

for chrom in sorted(df['chr1'].unique()):
    chrom_data = df[df['chr1'] == chrom]
    for link_type in ['Intrachromosomal', 'Interchromosomal']:
        type_data = chrom_data[chrom_data['link_type'] == link_type]
        if len(type_data) > 0:
            print(f"{chrom:<10} {link_type:<15} {len(type_data):<8} {type_data['score'].mean():<12.2f} {type_data['score'].std():<10.2f}")

# Overall statistics
print("\nOverall Statistics:")
print(f"Total intrachromosomal links: {len(df[df['link_type'] == 'Intrachromosomal'])}")
print(f"Total interchromosomal links: {len(df[df['link_type'] == 'Interchromosomal'])}")
print(f"Mean score - Intrachromosomal: {df[df['link_type'] == 'Intrachromosomal']['score'].mean():.2f}")
print(f"Mean score - Interchromosomal: {df[df['link_type'] == 'Interchromosomal']['score'].mean():.2f}")

In [ ]:
# Prepare data for seaborn
plot_data = merged_lengths.copy()

# Create the bar plot using seaborn
plt.figure(figsize=(10, 5))
ax = sns.barplot(x='chrom', y='length_mbp', hue='aln_type', data=plot_data,
                 palette={'Intrachromosomal': 'blue', 'Interchromosomal': 'red'},
                 order=natsorted(plot_data['chrom'].unique()))

# Remove top and right spines for publication style
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add horizontal guide lines
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)

# Customize the plot
plt.xlabel('Chromosome', fontsize=12)
plt.ylabel('Total Merged Alignment Length (Mbp)', fontsize=12)
plt.xticks(rotation=45)
plt.legend(title='Segdup Link Type', title_fontsize=10)

# Add value labels on the bars
for container in ax.containers:
    for bar in container:
        height = bar.get_height()
        if height > 0:  # Only label bars with positive height
            ax.text(bar.get_x() + bar.get_width() / 2, height + (ax.get_ylim()[1] * 0.01),
                   f'{height:.1f}', ha='center', va='bottom', fontsize=8, 
                    fontweight='bold', rotation=15)

plt.tight_layout()
plt.show()

# Print summary statistics (same as before)
print("Summary Statistics - Merged Alignment Lengths (Mbp):")
print("="*70)
print(f"{'Chromosome':<10} {'Intrachromosomal':<15} {'Interchromosomal':<15} {'Total':<10}")
print("-"*70)

for chrom in natsorted(pivot_lengths.index):
    intra_len = pivot_lengths.loc[chrom, 'Intrachromosomal']
    inter_len = pivot_lengths.loc[chrom, 'Interchromosomal']
    total_len = intra_len + inter_len
    print(f"{chrom:<10} {intra_len:<15.2f} {inter_len:<15.2f} {total_len:<10.2f}")

# Overall totals
total_intra = pivot_lengths['Intrachromosomal'].sum()
total_inter = pivot_lengths['Interchromosomal'].sum()
print("-"*70)
print(f"{'TOTAL':<10} {total_intra:<15.2f} {total_inter:<15.2f} {total_intra + total_inter:<10.2f}")

In [ ]:
# Calculate average genomic span per chromosome and segdup type
avg_genomic_span = flipped_df_for_counting.copy()
avg_genomic_span['genomic_span'] = avg_genomic_span['end1'] - avg_genomic_span['start1']
avg_genomic_span = avg_genomic_span.groupby(['chr1', 'link_type'])['genomic_span'].mean().reset_index()

# Create the plot
plt.figure(figsize=(12, 6))
ax = sns.barplot(x='chr1', y='genomic_span', hue='link_type', data=avg_genomic_span,
                 palette={'Intrachromosomal': 'blue', 'Interchromosomal': 'red'},
                 order=natsorted(avg_genomic_span['chr1'].unique()))

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add horizontal guide lines
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)

# Customize the plot
plt.xlabel('Chromosome', fontsize=12)
plt.ylabel('Average Genomic Span (bp)', fontsize=12)
plt.xticks(rotation=45)
plt.legend(title='Segdup Link Type', title_fontsize=10)

# Add value labels on the bars
for container in ax.containers:
    for bar in container:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, height + (ax.get_ylim()[1] * 0.01),
                   f'{int(height)}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Calculate average alignment length per chromosome and segdup type
avg_aln_length = flipped_df_for_counting.groupby(['chr1', 'link_type'])['aln_len'].mean().reset_index()

# Create the plot
plt.figure(figsize=(12, 6))
ax = sns.barplot(x='chr1', y='aln_len', hue='link_type', data=avg_aln_length,
                 palette={'Intrachromosomal': 'blue', 'Interchromosomal': 'red'},
                 order=natsorted(avg_aln_length['chr1'].unique()))

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add horizontal guide lines
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)

# Customize the plot
plt.xlabel('Chromosome', fontsize=12)
plt.ylabel('Average Alignment Length (bp)', fontsize=12)
plt.xticks(rotation=45)
plt.legend(title='Segdup Link Type', title_fontsize=10)

# Add value labels on the bars
for container in ax.containers:
    for bar in container:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, height + (ax.get_ylim()[1] * 0.01),
                   f'{int(height)}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

# Print summary statistics
print("AVERAGE ALIGNMENT LENGTH STATISTICS")
print("="*60)
print(f"{'Chromosome':<10} {'Link Type':<15} {'Avg Length (bp)':<15}")
print("-"*60)

for chrom in natsorted(avg_aln_length['chr1'].unique()):
    chrom_data = avg_aln_length[avg_aln_length['chr1'] == chrom]
    for link_type in ['Intrachromosomal', 'Interchromosomal']:
        type_data = chrom_data[chrom_data['link_type'] == link_type]
        if len(type_data) > 0:
            avg_length = type_data['aln_len'].values[0]
            print(f"{chrom:<10} {link_type:<15} {avg_length:<15.0f}")

print("-"*60)
# Overall averages
overall_intra = flipped_df_for_counting[flipped_df_for_counting['link_type'] == 'Intrachromosomal']['aln_len'].mean()
overall_inter = flipped_df_for_counting[flipped_df_for_counting['link_type'] == 'Interchromosomal']['aln_len'].mean()
print(f"{'OVERALL':<10} {'Intrachromosomal':<15} {overall_intra:<15.0f}")
print(f"{'OVERALL':<10} {'Interchromosomal':<15} {overall_inter:<15.0f}")

In [ ]:
size=15
# Create a single figure with three subplots
fig, axes = plt.subplots(3, 1, figsize=(10, 12))

# Get the natural sorted chromosome order for consistency
chrom_order = natsorted(flipped_df_for_counting['chr1'].unique())

# Plot 1: Count bar plot
ax1 = axes[0]
count_data = flipped_df_for_counting.groupby(['chr1', 'link_type']).size().reset_index(name='count')
sns.barplot(x='chr1', y='count', hue='link_type', data=count_data,
            palette={'Intrachromosomal': 'blue', 'Interchromosomal': 'red'},
            order=chrom_order, ax=ax1)

ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.yaxis.grid(True, linestyle='--', alpha=0.7)
ax1.set_xlabel('')  # Remove x-axis label
ax1.set_axisbelow(True)
# ax1.set_xlabel('Chromosome', fontsize=12)
ax1.set_ylabel('Number of Links', fontsize=size)
ax1.set_xticklabels([])#ax1.get_xticklabels(), rotation=45)
# ax1.set_title('A) Segdup Count by Chromosome', fontsize=12, fontweight='bold')
ax1.tick_params(axis='x', which='both', bottom=False)  # Remove x-ticks completely
# Add count labels
for container in ax1.containers:
    for bar in container:
        height = bar.get_height()
        if height > 0:
            ax1.text(bar.get_x() + bar.get_width() / 2, height + (ax1.get_ylim()[1] * 0.01),
                   f'{int(height)}', ha='center', va='bottom', fontsize=7, fontweight='bold', rotation=15)

# Plot 2: Violin plot
ax2 = axes[1]
sns.violinplot(x='chr1', y='score', hue='link_type', data=flipped_df_for_counting, 
               palette={'Intrachromosomal': 'blue', 'Interchromosomal': 'red'},
               split=True, inner='quartile', edgecolor="black", order=chrom_order, ax=ax2,
               hue_order=['Interchromosomal', 'Intrachromosomal'])  # ADD THIS LINE

ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.yaxis.grid(True, linestyle='--', alpha=0.7)
ax2.set_axisbelow(True)
ax2.set_xlabel('')  # Remove x-axis label
ax2.set_ylabel('Total Alignment Error', fontsize=size)
ax2.set_xticklabels([])  # Remove x-tick labels
ax2.tick_params(axis='x', which='both', bottom=False)  # Remove x-ticks completely

# Plot 3: Merged length bar plot
ax3 = axes[2]
sns.barplot(x='chrom', y='length_mbp', hue='aln_type', data=plot_data,
            palette={'Intrachromosomal': 'blue', 'Interchromosomal': 'red'},
            order=chrom_order, ax=ax3)

ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
ax3.yaxis.grid(True, linestyle='--', alpha=0.7)
ax3.set_axisbelow(True)
ax3.set_xlabel('Chromosome', fontsize=size)
ax3.set_ylabel('Total Merged Alignment Length (Mbp)', fontsize=size)
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=45)
# ax3.set_title('C) Merged Length by Chromosome', fontsize=12, fontweight='bold')

# Add length labels
for container in ax3.containers:
    for bar in container:
        height = bar.get_height()
        if height > 0:
            ax3.text(bar.get_x() + bar.get_width() / 2, height + (ax3.get_ylim()[1] * 0.01),
                   f'{height:.1f}', ha='center', va='bottom', fontsize=7, fontweight='bold', rotation=15)

# Remove individual legends and create one shared legend
for ax in axes:
    ax.get_legend().remove()

# Create a single shared legend
handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, title='Segmental Duplication Type', title_fontsize=10,
           loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=False)
fig.align_ylabels(axes)

plt.tight_layout()
plt.subplots_adjust(bottom=0.0)  # Make room for the shared legend
plt.show()

# Print combined summary statistics
print("COMBINED SUMMARY STATISTICS")
print("="*100)

# Count statistics
print("\n1. COUNT STATISTICS:")
print("-"*60)
print(f"{'Chromosome':<10} {'Intra Count':<12} {'Inter Count':<12} {'Total Count':<12}")
print("-"*60)
for chrom in chrom_order:
    chrom_data = flipped_df_for_counting[flipped_df_for_counting['chr1'] == chrom]
    intra_count = len(chrom_data[chrom_data['link_type'] == 'Intrachromosomal'])
    inter_count = len(chrom_data[chrom_data['link_type'] == 'Interchromosomal'])
    total_count = intra_count + inter_count
    print(f"{chrom:<10} {intra_count:<12} {inter_count:<12} {total_count:<12}")

# Score statistics
print(f"\n2. SCORE STATISTICS:")
print("-"*60)
print(f"{'Link Type':<15} {'Count':<8} {'Mean Score':<12} {'Std Dev':<10}")
print("-"*60)
for link_type in ['Intrachromosomal', 'Interchromosomal']:
    type_data = flipped_df_for_counting[flipped_df_for_counting['link_type'] == link_type]
    if len(type_data) > 0:
        print(f"{link_type:<15} {len(type_data):<8} {type_data['score'].mean():<12.2f} {type_data['score'].std():<10.2f}")

# Length statistics
print(f"\n3. MERGED LENGTH STATISTICS (Mbp):")
print("-"*60)
print(f"{'Link Type':<15} {'Total Length':<15}")
print("-"*60)
print(f"{'Intrachromosomal':<15} {pivot_lengths['Intrachromosomal'].sum():<15.2f}")
print(f"{'Interchromosomal':<15} {pivot_lengths['Interchromosomal'].sum():<15.2f}")
print(f"{'TOTAL':<15} {(pivot_lengths['Intrachromosomal'].sum() + pivot_lengths['Interchromosomal'].sum()):<15.2f}")

In [ ]:
# Total segdup bp per chromosome (sum of merged intra + inter lengths), then merge with
# chromosome sizes and express as a percentage of each chromosome.
result_df = merged_lengths.groupby('chrom')['length'].sum().reset_index()
result_df.columns = ['Chromosome', 'Total_BP']

result_df = fai.merge(result_df, how='left', left_on='chr', right_on='Chromosome')
result_df['segdup_percentage'] = (result_df['Total_BP'] / result_df['length']) * 100
result_df['non_segdup_percentage'] = 100 - result_df['segdup_percentage']


In [ ]:

# Create the stacked barplot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Stacked bar plot (percentages)
bars1 = ax1.bar(result_df['Chromosome'], result_df['segdup_percentage'], 
                label='Segmental Duplications', color='red', alpha=0.7)
bars2 = ax1.bar(result_df['Chromosome'], result_df['non_segdup_percentage'], 
                bottom=result_df['segdup_percentage'], 
                label='Non-duplicated', color='lightblue', alpha=0.7)

ax1.set_ylabel('Percentage of Chromosome')
ax1.set_title('Percentage of Each Chromosome Covered by Segmental Duplications')
ax1.legend()

# Add percentage labels on bars
for bar, percentage in zip(bars1, result_df['segdup_percentage']):
    height = bar.get_height()
    if height > 5:  # Only label if segment is large enough
        ax1.text(bar.get_x() + bar.get_width()/2., height/2,
                f'{percentage:.1f}%', ha='center', va='center', 
                fontweight='bold', color='black',rotation=90)

# Plot 2: Actual basepairs
ax2.bar(result_df['Chromosome'], result_df['Total_BP'] / 1e6, 
        color='red', alpha=0.7, label='Segmental Duplications')
ax2.set_ylabel('Segdup Coverage (Mbp)')
ax2.set_xlabel('Chromosome')
ax2.set_title('Total Segmental Duplication Coverage per Chromosome (Mbp)')

# Add value labels on top of bars
for i, (chrom, bp) in enumerate(zip(result_df['Chromosome'], result_df['Total_BP'])):
    ax2.text(i, (bp / 1e6) + 0.1, f'{bp/1e6:.1f}M', 
             ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# Print summary table
print("Segmental Duplication Coverage by Chromosome:")
print("="*60)
print(f"{'Chromosome':<10} {'Length (Mbp)':<12} {'Segdup (Mbp)':<12} {'Percentage':<10}")
print("-"*60)
for _, row in result_df.iterrows():
    print(f"{row['Chromosome']:<10} {row['length']/1e6:<12.1f} {row['Total_BP']/1e6:<12.1f} {row['segdup_percentage']:<10.2f}%")

print("-"*60)
total_length = result_df['length'].sum()
total_segdup = result_df['Total_BP'].sum()
total_percentage = (total_segdup / total_length) * 100
print(f"{'TOTAL':<10} {total_length/1e6:<12.1f} {total_segdup/1e6:<12.1f} {total_percentage:<10.2f}%")